<a href="https://colab.research.google.com/github/smruthi215/DLG--LAB-1/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# XOR input and output
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])  # Inputs
y = np.array([[0], [1], [1], [0]])  # Outputs

# Build the MLP model
model = Sequential()
model.add(Dense(8, input_dim=2, activation='relu'))  # Hidden layer with 8 neurons
model.add(Dense(1, activation='sigmoid'))  # Output layer with 1 neuron for binary output

# Compile the model
model.compile(
    loss='binary_crossentropy',
    optimizer=SGD(learning_rate=0.1),
    metrics=['accuracy']
)

# Train the model
model.fit(X, y, epochs=1000, verbose=0)

# Evaluate the model
_, accuracy = model.evaluate(X, y)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Make predictions
predictions = model.predict(X)
print("Predictions:")
print(np.round(predictions))  # Rounding to 0 or 1

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 1.0000 - loss: 0.0148
Model Accuracy: 100.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Predictions:
[[0.]
 [1.]
 [1.]
 [0.]]


In [4]:
import numpy as np
from sklearn.datasets import make_moons

# ----- Data -----
X, y = make_moons(n_samples=400, noise=0.20, random_state=1)
y = y.reshape(-1, 1).astype(float)
X = (X - X.mean(0)) / X.std(0)          # standardize


# ----- Helpers -----
def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def sig(z):
    return 1 / (1 + np.exp(-z))


def init(sizes, seed=0):
    rng = np.random.default_rng(seed)
    P = []

    for a, b in zip(sizes[:-1], sizes[1:]):
        W = rng.normal(0, np.sqrt(2/a), size=(a, b))   # He initialization
        P.append([W, np.zeros((1, b))])

    return P


def grads(P, Xb, yb):
    # forward + backward on a batch
    zs, acts, a = [], [Xb], Xb

    for i, (W, b) in enumerate(P):
        z = a @ W + b
        zs.append(z)

        a = sig(z) if i == len(P)-1 else relu(z)
        acts.append(a)

    m = Xb.shape[0]
    g = [None] * len(P)

    dz = (acts[-1] - yb) / m          # dL/dz = yhat - y
                                       # (sigmoid+BCE)

    for i in reversed(range(len(P))):
        g[i] = [
            acts[i].T @ dz,
            dz.sum(0, keepdims=True)
        ]

        if i > 0:
            dz = (dz @ P[i][0].T) * drelu(zs[i-1])

    return g


def accuracy(P, X, y):
    a = X

    for i, (W, b) in enumerate(P):
        a = sig(a @ W + b) if i == len(P)-1 else relu(a @ W + b)

    return np.mean((a > 0.5) == (y > 0.5))


sizes, lr, EPOCHS = [2, 16, 16, 1], 0.5, 200


# ----- (A) Batch Gradient Descent: ALL data, 1 update / epoch -----
P = init(sizes, seed=0)

for epoch in range(EPOCHS):
    g = grads(P, X, y)

    for k in range(len(P)):
        P[k][0] -= lr * g[k][0]
        P[k][1] -= lr * g[k][1]

print("Batch GD accuracy:", accuracy(P, X, y))


# ----- (B) Stochastic Gradient Descent: mini-batches, N/B updates / epoch -----
P = init(sizes, seed=0)
B = 16
idx = np.arange(len(X))

for epoch in range(EPOCHS):
    np.random.shuffle(idx)                  # shuffle every epoch!

    for s in range(0, len(X), B):
        b = idx[s:s+B]

        g = grads(P, X[b], y[b])

        for k in range(len(P)):
            P[k][0] -= lr * g[k][0]
            P[k][1] -= lr * g[k][1]

print("SGD accuracy:", accuracy(P, X, y))

Batch GD accuracy: 0.965
SGD accuracy: 0.975


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD

model = Sequential([
    Input(shape=(2,)),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=SGD(learning_rate=0.5),
    metrics=['accuracy']
)

# Batch Gradient Descent → use the whole dataset as one batch
model.fit(X, y, epochs=200, batch_size=len(X), verbose=0)

# Stochastic / mini-batch → small batch size, many updates per epoch
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

In [2]:
import numpy as np

def sigmoid ( x ) :
    return 1 / (1 + np . exp ( - x ) )


def sigmoid_derivative ( x ) :
    return x * (1 - x )

inputs=np . array ([[0 , 0] , [0 , 1] , [1 , 0] , [1 , 1]])
expected_output = np . array ([[0] , [1] , [1] , [0]])

inputLayerNeurons , hiddenLayerNeurons , outputNeurons = 2 , 2 , 1
hidden_weights = np . random . uniform ( size =( inputLayerNeurons , hiddenLayerNeurons ) )
hidden_bias = np . random . uniform ( size =(1 , hiddenLayerNeurons ) )
output_weights = np . random . uniform ( size =( hiddenLayerNeurons , outputNeurons ) )
output_bias = np . random . uniform ( size =(1 , outputNeurons ) )

lr = 0.1

epochs = 10000
for _ in range ( epochs ) :

    hidden_layer_activation = np . dot ( inputs , hidden_weights ) + hidden_bias
    hidden_layer_output = sigmoid ( hidden_layer_activation )

    output_layer_activation = np . dot ( hidden_layer_output , output_weights )+ output_bias

    predicted_output = sigmoid ( output_layer_activation )
    error = expected_output - predicted_output
    d_predicted_output = error * sigmoid_derivative ( predicted_output )

    error_hidden_layer = d_predicted_output . dot ( output_weights . T )
    d_hidden_layer = error_hidden_layer * sigmoid_derivative (hidden_layer_output )

    output_weights += hidden_layer_output . T . dot ( d_predicted_output ) * lr
    output_bias += np . sum( d_predicted_output , axis =0 , keepdims = True ) * lr
    hidden_weights += inputs . T . dot ( d_hidden_layer ) * lr
    hidden_bias += np . sum( d_hidden_layer , axis =0 , keepdims = True ) * lr


print (" Predicted Output :")
print ( predicted_output )

 Predicted Output :
[[0.15925673]
 [0.8178845 ]
 [0.81789291]
 [0.21953623]]


In [ ]:
https://github.com/smruthi215/DLG--LAB-1